# 🧠 CerebroSpheroid V5: Google Gemma 4 E4B to Cerebro Distillation
### Google Gemma 4 E4B Teacher • CerebroSpheroid V5 Small (105.9M) Student • Option B Full Soft Distillation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/gemma2cerebro.ipynb)

- **Teacher Model**: Google Gemma 4 E4B (`google/gemma-4-E4B-it` / `google/gemma-4-E4B-it-qat-q4_0-gguf`). Loaded in 4-bit NormalFloat (BitsAndBytes) requiring only ~2.5 - 3.0 GB VRAM.
- **Student Model**: CerebroSpheroid V5 Small (105.9M parameters, 16 cortical shells, 40 shortcut tracts, 32,000 vocab).
- **Option B Full Soft-Target Logit Distillation**: Cross-Tokenizer Top-$K$ ($K=64$) projection mapping Gemma's 262,144 probability spread onto Cerebro's 32,000 vocabulary, scaling dark knowledge with temperature $T=2.0$ and dual loss $\mathcal{L}_{\text{total}} = (1 - \alpha)\mathcal{L}_{\text{hard}} + \alpha \cdot T^2 \cdot \mathcal{L}_{\text{soft}}$.
- **Answer-Anchor Loss Weighting (AALW)**: 0x on prompt, 3.5x on deliberative thoughts (`<think> ... </think>`), 4.0x on tool actions (`<act> ... </act>`), and 5.0x on final answer tokens.
- **Direct Website Cloud Mirroring**: Checkpoints saved locally as `gemma2cerebro.pt` and synchronized directly to your website (`https://bioflo.in`) in non-blocking background threads. Zero Google Drive dependency; runs seamlessly on Kaggle and Google Colab.

---
### ⚡ Hardware Setting:
- **Google Colab**: Runtime -> Change runtime type -> **T4 GPU** (or A100 / L4 GPU)
- **Kaggle**: Settings -> Accelerator -> **GPU T4 x2** (or GPU P100)

## ⚡ Step 1: System Dependencies, Hugepages & Repository Sync
Installs dependencies, detects GPU acceleration, and pulls the latest codebase from GitHub (universal for Kaggle and Google Colab).

In [ ]:
# ==============================================================================
# 🚀 Step 1: System Dependencies, Hugepages & Repository Sync (Kaggle & Colab)
# ==============================================================================
!echo always 2>/dev/null | sudo -n tee /sys/kernel/mm/transparent_hugepage/enabled >/dev/null 2>&1 || true
!pip install -q transformers datasets sentencepiece accelerate bitsandbytes huggingface_hub torch

import os, sys, subprocess, importlib, torch

# 1. Verify GPU Acceleration
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"⚡ CUDA GPU Active: {dev_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ Running on CPU. In Colab/Kaggle, enable GPU acceleration.")

# 2. Universal Environment & Repository Resolution (Kaggle & Colab)
IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
REPO_DIR = "/kaggle/working/cerebro_v5" if IS_KAGGLE else "/content/cerebro_v5"

if not os.path.exists(REPO_DIR):
    print(f"📥 Cloning cerebro_spheroid_v5 repository to {REPO_DIR}...")
    subprocess.run(["git", "clone", "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git", REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Convenience symlinks: ensure running from either /kaggle/working OR /kaggle/working/cerebro_v5 works 100%
if IS_KAGGLE:
    for item in ["colab", "src", "data", "training"]:
        src_p = os.path.join(REPO_DIR, item)
        dst_p = os.path.join("/kaggle/working", item)
        if os.path.exists(src_p) and not os.path.exists(dst_p):
            try:
                os.symlink(src_p, dst_p)
            except Exception:
                pass

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Change Jupyter shell directory magic to REPO_DIR
try:
    get_ipython().run_line_magic("cd", REPO_DIR)
except Exception:
    pass

# Clear module caches
importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith(("src", "training", "autotune", "colab")):
        del sys.modules[mod]

# 3. Scratch Storage Setup (Zero Google Drive Dependency)
LOCAL_CKPT_DIR = "/kaggle/working/checkpoints" if IS_KAGGLE else "/content/checkpoints"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
print(f"📁 Local Checkpoint Scratch Storage: {LOCAL_CKPT_DIR}")
print("🌐 Checkpoints are automatically synced to https://bioflo.in as gemma2cerebro.pt (Zero Google Drive needed!).\n")

## ⚡ Step 2: Configure & Launch Option B Distillation
Executes teacher-to-student distillation using Google Gemma 4 E4B with **Option B Full Soft-Target Logit Distillation** (Top-64 cross-tokenizer projection, $T=2.0$, $\alpha=0.6$), Answer-Anchor Loss Weighting, and automatic background upload to your website (`https://bioflo.in`) as `gemma2cerebro.pt`.

In [ ]:
# ==============================================================================
# ⚡ Step 2: Launch Google Gemma 4 E4B -> Cerebro V5 Small Distillation (Option B)
# ==============================================================================
from colab.distill_gemma_to_v5_small import run_distillation

# Primary Distillation Hyperparameters (Option B: Full Soft-Target Distribution)
TEACHER_MODEL   = "google/gemma-4-E4B-it" # Google Gemma 4 E4B (or "google/gemma-4-E4B-it-qat-q4_0-gguf")
DISTILL_MODE    = "stream"                # "stream" (fast frontier dataset), "live" (on-policy), "hybrid"
TOTAL_STEPS     = 5000                    # Total distillation steps
BATCH_SIZE      = 2                       # Micro-batch size per step (zero-OOM on dual T4)
GRAD_ACCUM      = 8                       # Gradient accumulation (Effective batch = 16)
LEARNING_RATE   = 3e-4                    # Peak learning rate with cosine decay
AALW_WEIGHT     = 4.0                     # Loss weight on deliberation & answer tokens
SOFT_DISTILL    = True                    # Option B: Full soft-target probability distribution distillation
DISTILL_TEMP    = 2.0                     # Distillation temperature T (softens logits for dark knowledge)
DISTILL_ALPHA   = 0.6                     # Loss balance: alpha * T^2 * L_soft + (1 - alpha) * L_hard
TOPK            = 64                      # Cross-tokenizer top-k probability projection
SAVE_INTERVAL   = 250                     # Steps per checkpoint save & website sync
LOAD_IN_4BIT    = True                    # 4-bit quantization (fits Gemma 4 in ~2.5 - 3.0 GB VRAM)

run_distillation(
    teacher_model_id=TEACHER_MODEL,
    student_preset="v5_small",
    mode=DISTILL_MODE,
    total_steps=TOTAL_STEPS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    aalw_weight=AALW_WEIGHT,
    soft_distill=SOFT_DISTILL,
    distill_temp=DISTILL_TEMP,
    distill_alpha=DISTILL_ALPHA,
    topk=TOPK,
    save_interval=SAVE_INTERVAL,
    cloud_sync=True,
    load_in_4bit=LOAD_IN_4BIT,
)


## ⚡ Step 3: Interactive Deliberation & Reasoning Probe
Test prompt completions from the distilled student model (`gemma2cerebro.pt`) to inspect generated `<think> ... </think>` reasoning and answers. If running on a fresh session, it automatically pulls `gemma2cerebro.pt` from your website (`https://bioflo.in`).

In [ ]:
# ==============================================================================
# 🔍 Step 3: Interactive Prompt Testing on Distilled V5 Small (gemma2cerebro.pt)
# ==============================================================================
import torch, os
from transformers import AutoTokenizer
from src.spheroid_v5 import CerebroSpheroidV5, ModelConfigV5, PRESETS_V5

device = torch.device("cuda:1" if torch.cuda.device_count() >= 2 else ("cuda:0" if torch.cuda.is_available() else "cpu"))
tok_dir = os.path.join(REPO_DIR, "data", "tokenizer")
tokenizer = AutoTokenizer.from_pretrained(tok_dir if os.path.exists(tok_dir) else "meta-llama/Llama-2-7b-hf")

# 1. Locate Local Checkpoint or Auto-Pull from bioflo.in
IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ckpt_candidates = [
    "/kaggle/working/checkpoints/gemma2cerebro.pt",
    "/content/checkpoints/gemma2cerebro.pt",
    os.path.join(REPO_DIR, "checkpoints/gemma2cerebro.pt"),
    os.path.join(REPO_DIR, "checkpoints/distill_gemma_v5_small/gemma2cerebro.pt"),
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)

if not ckpt_path:
    print("🌐 Local checkpoint not found. Pulling gemma2cerebro.pt from bioflo.in...")
    target_dest = "/kaggle/working/checkpoints/gemma2cerebro.pt" if IS_KAGGLE else "/content/checkpoints/gemma2cerebro.pt"
    try:
        from src.cloud_sync import pull_checkpoint
        if pull_checkpoint(local_dest=target_dest, model="gemma2cerebro", target_type="latest"):
            ckpt_path = target_dest
    except Exception as e:
        print(f"ℹ️ Could not pull from bioflo.in: {e}")

cfg = ModelConfigV5(**PRESETS_V5["v5_small"])
model = CerebroSpheroidV5(cfg).to(device)
if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt.get("model_state", ckpt), strict=False)
    print(f"✅ Loaded checkpoint from {ckpt_path} (Step {ckpt.get('step', '?')})")
model.eval()

def generate_reasoning(prompt_text, max_tokens=128):
    formatted = f"<stim> {prompt_text.strip()} </stim>"
    input_ids = tokenizer.encode(formatted, add_special_tokens=False)
    p_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    out_tokens = []
    with torch.no_grad():
        for _ in range(max_tokens):
            tgt_in = torch.tensor([[1] + out_tokens], dtype=torch.long, device=device) if out_tokens else torch.tensor([[1]], dtype=torch.long, device=device)
            _, logits = model(prompts=p_tensor, targets=tgt_in, deliberation_mode="teacher_forcing")
            next_id = int(torch.argmax(logits[0, -1, :]).item())
            if next_id in (0, 2):
                break
            out_tokens.append(next_id)
    return tokenizer.decode(out_tokens, skip_special_tokens=True)

test_prompt = "If a cell divides every 20 minutes, how many cells will be produced from 1 cell after 2 hours?"
print(f"Prompt:   {test_prompt}")
print(f"Response: {generate_reasoning(test_prompt)}")
